---
## 🏆 도전 과제

지금까지 배운 내용을 활용해 **영화 리뷰 분석**을 완성해보세요!  
막히면 위의 CH 8 코드를 참고하세요.

**목표:**
1. 영화 리뷰 5개를 파이프라인으로 전처리
2. 단어를 모두 합쳐서 빈도 분석
3. TOP 5 단어 출력

In [ ]:
# === 과제용 영화 리뷰 데이터 ===
# 실제 네이버, 다음 영화 리뷰 10개를 추출해서 거기에 대한 리뷰 분석
# 코드에 맞게 실행해보고 결론이 무엇인지? 한계점이 무엇인지 정리해서 제출
# 코드 수행 + 느낀점 코드 형태로 적어서 제출

In [4]:
# KoNLPy는 내부적으로 Java를 사용합니다
# Java를 먼저 설치해야 KoNLPy가 작동합니다
!apt-get install default-jdk -y

# konlpy : 한국어 형태소 분석 라이브러리
# nltk   : 영어 자연어 처리 라이브러리
# --quiet : 설치 메시지를 간략하게 출력
!pip install konlpy nltk --quiet

# 설치 완료 메시지
print("✅ 설치 완료!")

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  at-spi2-core default-jdk-headless default-jre default-jre-headless
  fonts-dejavu-core fonts-dejavu-extra gsettings-desktop-schemas
  libatk-bridge2.0-0 libatk-wrapper-java libatk-wrapper-java-jni libatk1.0-0
  libatk1.0-data libatspi2.0-0 libxcomposite1 libxt-dev libxtst6 libxxf86dga1
  openjdk-11-jdk openjdk-11-jdk-headless openjdk-11-jre
  openjdk-11-jre-headless session-migration x11-utils
Suggested packages:
  libxt-doc openjdk-11-demo openjdk-11-source visualvm libnss-mdns
  fonts-ipafont-gothic fonts-ipafont-mincho fonts-wqy-microhei
  | fonts-wqy-zenhei fonts-indic mesa-utils
The following NEW packages will be installed:
  at-spi2-core default-jdk default-jdk-headless default-jre
  default-jre-headless fonts-dejavu-core fonts-dejavu-extra
  gsettings-desktop-schemas libatk-bridge2.0-0 libatk-wrapper-java
  libatk-wrapper-java-jn

In [5]:
# ── 라이브러리 불러오기 ──────────────────────

import re                    # 정규표현식 : 텍스트에서 특정 패턴을 찾거나 바꿀 때 사용
from collections import Counter  # Counter : 단어 빈도를 쉽게 셀 때 사용

import nltk                             # 영어 자연어 처리 도구 모음
nltk.download('punkt',     quiet=True)  # 문장/단어 토큰화 모델
nltk.download('punkt_tab', quiet=True)  # 토큰화 추가 데이터
nltk.download('stopwords', quiet=True)  # 영어 불용어 목록
nltk.download('wordnet',   quiet=True)  # 영어 단어 원형 데이터

from konlpy.tag import Okt   # Okt : KoNLPy에서 가장 많이 쓰이는 한국어 형태소 분석기
okt = Okt()                  # 형태소 분석기 객체 생성 (한 번만 만들면 계속 사용 가능)

print("✅ 준비 완료! 이제 CH 1 부터 시작하세요 🚀")

✅ 준비 완료! 이제 CH 1 부터 시작하세요 🚀


In [1]:
# ── 전처리 파이프라인 함수 정의 ─────────────────
# 모든 단계를 하나로 묶은 함수입니다

# 분석에 불필요한 한국어 단어 목록
불용어_목록 = [
    "것", "수", "나", "저", "제", "그", "이", "때",
    "등", "좀", "잘", "더", "한", "안", "못", "또"
]

def 전처리_파이프라인(텍스트):
    """
    입력: 원본 텍스트 (문자열)
    출력: 전처리된 핵심 단어 리스트
    """
    # 1단계: 정제 — 노이즈 제거
    텍스트 = re.sub(r'http\S+', '', 텍스트)               # URL 제거
    텍스트 = re.sub(r'<[^>]+>', '', 텍스트)               # HTML 태그 제거
    텍스트 = re.sub(r'[^가-힣a-zA-Z0-9 ]', '', 텍스트)    # 특수문자 제거
    텍스트 = re.sub(r'\s+', ' ', 텍스트).strip()           # 공백 정리

    # 2단계: 정규화 — 반복 문자 압축
    텍스트 = re.sub(r'(.)\1{2,}', r'\1\1', 텍스트)

    # 3단계: 형태소 분석 — 명사 추출
    토큰들 = okt.nouns(텍스트)

    # 4단계: 불용어 제거 + 2글자 이상만 유지
    결과 = [
        단어 for 단어 in 토큰들
        if 단어 not in 불용어_목록  # 불용어 목록에 없는 것만
        and len(단어) >= 2          # 2글자 이상인 것만
    ]

    return 결과

print("✅ 파이프라인 함수 준비 완료!")

✅ 파이프라인 함수 준비 완료!


In [6]:
# ── 과제용 영화 리뷰 데이터 ──────────────────

영화리뷰1 = "개인적으로는 7번방보다 이영화가 더 나은듯"
영화리뷰2 = "재미있는 상황 연출과 소소한 유머 그리고 이병헌의 미친듯한 연기. 이병헌의 연기는 경악하고 소름끼칠 정도."
영화리뷰3 = "이병헌씨 연기력이 대단해요줄거리도 탄탄하고 ^^너무 감동 받았읍니다"
영화리뷰4 = "소름쫙 3번을 봐도 영화가 전달해주고픈 메세지와 그 영화 자체에 소름이 끼친다 정말 남녀노소 봐야될 대표한국영화!!!"
영화리뷰5 = "베를린따위같은헐리우드아류작보다는 광해가 오천만배는 나은듯...영화속 광해같은지도자는 과연영화속에서만존재할수밖에없는것인가 ㅠㅠ"

print("영화 리뷰 5개 준비 완료!")
print()
print("1:", 영화리뷰1)
print("2:", 영화리뷰2)
print("3:", 영화리뷰3)
print("4:", 영화리뷰4)
print("5:", 영화리뷰5)

영화 리뷰 5개 준비 완료!

1: 개인적으로는 7번방보다 이영화가 더 나은듯
2: 재미있는 상황 연출과 소소한 유머 그리고 이병헌의 미친듯한 연기. 이병헌의 연기는 경악하고 소름끼칠 정도.
3: 이병헌씨 연기력이 대단해요줄거리도 탄탄하고 ^^너무 감동 받았읍니다
4: 소름쫙 3번을 봐도 영화가 전달해주고픈 메세지와 그 영화 자체에 소름이 끼친다 정말 남녀노소 봐야될 대표한국영화!!!
5: 베를린따위같은헐리우드아류작보다는 광해가 오천만배는 나은듯...영화속 광해같은지도자는 과연영화속에서만존재할수밖에없는것인가 ㅠㅠ


In [7]:
# ✏️ STEP 1: 각 리뷰를 파이프라인으로 전처리하세요
# 힌트: 전처리_파이프라인(리뷰) 형태로 사용합니다

# 리뷰1 전처리
전처리1 = 전처리_파이프라인(영화리뷰1)
print("리뷰1 전처리:", 전처리1)

# 리뷰2 전처리
전처리2 = 전처리_파이프라인(영화리뷰2)
print("리뷰2 전처리:", 전처리2)

# 리뷰3 전처리
전처리3 = 전처리_파이프라인(영화리뷰3)
print("리뷰3 전처리:", 전처리3)

# 리뷰4 전처리
전처리4 = 전처리_파이프라인(영화리뷰4)
print("리뷰4 전처리:", 전처리4)

# 리뷰5 전처리
전처리5 = 전처리_파이프라인(영화리뷰5)
print("리뷰5 전처리:", 전처리5)

리뷰1 전처리: ['개인', '번방', '이영화']
리뷰2 전처리: ['상황', '연출', '유머', '이병헌', '연기', '이병헌', '연기', '경악', '정도']
리뷰3 전처리: ['이병헌', '연기력', '줄거리', '감동', '읍니']
리뷰4 전처리: ['소름', '영화', '전달', '메세지', '영화', '자체', '소름', '정말', '남녀', '노소', '대표', '한국영']
리뷰5 전처리: ['베를린', '따위', '헐리우드', '류작', '광해', '오천만배', '영화', '광해', '지도자', '과연', '영화', '존재']


In [8]:
# ✏️ STEP 2: 5개 리뷰의 단어를 하나의 리스트로 합치세요
# 힌트: extend() 를 사용합니다

영화_단어들 = []  # 빈 리스트 준비

영화_단어들.extend(전처리1)  # 리뷰1 단어 추가
영화_단어들.extend(전처리2)  # 리뷰2 단어 추가
영화_단어들.extend(전처리3)  # 리뷰3 단어 추가
영화_단어들.extend(전처리4)  # 리뷰4 단어 추가
영화_단어들.extend(전처리5)  # 리뷰5 단어 추가

print("전체 단어 목록:", 영화_단어들)
print("총 단어 수:", len(영화_단어들), "개")

전체 단어 목록: ['개인', '번방', '이영화', '상황', '연출', '유머', '이병헌', '연기', '이병헌', '연기', '경악', '정도', '이병헌', '연기력', '줄거리', '감동', '읍니', '소름', '영화', '전달', '메세지', '영화', '자체', '소름', '정말', '남녀', '노소', '대표', '한국영', '베를린', '따위', '헐리우드', '류작', '광해', '오천만배', '영화', '광해', '지도자', '과연', '영화', '존재']
총 단어 수: 41 개


In [9]:
# ✏️ STEP 3: 단어 빈도를 계산하고 TOP 5를 출력하세요
# 힌트: Counter() 와 .most_common(5) 를 사용합니다

# Counter로 빈도 계산
영화_빈도 = Counter(영화_단어들)

print("📊 영화 리뷰 TOP 5 키워드")
print()

# 순위별로 출력
for 순위, (단어, 횟수) in enumerate(영화_빈도.most_common(5), 1):
    별 = "★" * 횟수  # 횟수만큼 별 표시
    print(f"  {순위}위: {단어} ({횟수}회) {별}")

📊 영화 리뷰 TOP 5 키워드

  1위: 영화 (4회) ★★★★
  2위: 이병헌 (3회) ★★★
  3위: 연기 (2회) ★★
  4위: 소름 (2회) ★★
  5위: 광해 (2회) ★★


In [10]:
# 🔥 도전! 불용어를 추가해서 결과를 개선해보세요
#
# TOP 5 결과를 보고 분석에 불필요한 단어가 있으면
# 아래 추가_불용어 목록에 넣고 다시 실행해보세요!

# ↓ 여기에 추가하고 싶은 단어를 넣어보세요
추가_불용어 = [
    # 예시: "주의", "생각", "올해"
    "읍니", "한국영", "남녀", "노소"
]

# 기존 불용어 + 새 불용어 합치기
# + 연산자 : 두 리스트를 이어붙입니다
확장_불용어 = 불용어_목록 + 추가_불용어

# 확장된 불용어로 다시 전처리하는 함수
def 전처리_파이프라인_v2(텍스트):
    텍스트 = re.sub(r'http\S+', '', 텍스트)
    텍스트 = re.sub(r'[^가-힣a-zA-Z0-9 ]', '', 텍스트)
    텍스트 = re.sub(r'(.)\1{2,}', r'\1\1', 텍스트)
    텍스트 = re.sub(r'\s+', ' ', 텍스트).strip()
    토큰들 = okt.nouns(텍스트)
    # 확장된 불용어 목록 사용
    결과 = [t for t in 토큰들 if t not in 확장_불용어 and len(t) >= 2]
    return 결과


# v2로 다시 전처리
개선_단어들 = []
개선_단어들.extend(전처리_파이프라인_v2(영화리뷰1))
개선_단어들.extend(전처리_파이프라인_v2(영화리뷰2))
개선_단어들.extend(전처리_파이프라인_v2(영화리뷰3))
개선_단어들.extend(전처리_파이프라인_v2(영화리뷰4))
개선_단어들.extend(전처리_파이프라인_v2(영화리뷰5))

개선_빈도 = Counter(개선_단어들)

print("📊 개선 후 TOP 5")
for 순위, (단어, 횟수) in enumerate(개선_빈도.most_common(5), 1):
    print(f"  {순위}위: {단어} ({횟수}회)")

📊 개선 후 TOP 5
  1위: 영화 (4회)
  2위: 이병헌 (3회)
  3위: 연기 (2회)
  4위: 소름 (2회)
  5위: 광해 (2회)
